# 02. Huấn Luyện & Đánh Giá Mô Hình SVM Hai Tầng (Training & Evaluation)

Quy trình xây dựng mô hình Support Vector Machine (SVM) trên đặc trưng HOG:
1. Trích xuất đặc trưng HOG (Histogram of Oriented Gradients) kích thước 1.764 chiều.
2. Huấn luyện Tầng 1: Binary SVM lọc nền (Sign vs Background).
3. Huấn luyện Tầng 2: Multiclass SVM phân loại 52 lớp biển báo.
4. Tinh chỉnh siêu tham số $C$ và $\gamma$ bằng GridSearchCV với Stratified K-Fold CV.
5. Đánh giá: Macro-F1, Accuracy, Ma trận nhầm lẫn và Top các cặp lớp dễ nhầm.
6. Lưu trữ mô hình vào thư mục `outputs/models/`.

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.metrics import classification_report

from src.feature_extraction import extract_hog_features
from src.classifier import train_svm, tune_svm, evaluate_svm, analyze_confusion_and_per_class_f1, save_model
from src.data_loader import load_config, load_image
from src.utils import read_label_boxes

## 1. Kiểm Thử Trích Xuất Vector HOG 1.764 Chiều

Với ảnh ROI chuẩn hóa $64 \times 64$, cell $8 \times 8$, block $2 \times 2$, 9 bins hướng gradient:
$$\text{Số chiều} = (8 - 1) \times (8 - 1) \times (2 \times 2 \times 9) = 49 \times 36 = 1.764$$

In [ ]:
dummy_roi = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
feat, hog_img = extract_hog_features(dummy_roi, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
print(f"Độ dài vector HOG: {feat.shape[0]} chiều (Kỳ vọng: 1764)")

## 2. Huấn Luyện & Đánh Giá Pipeline Phân Loại

In [ ]:
# Tạo dữ liệu mô phỏng cho kiểm thử pipeline
rng = np.random.default_rng(42)
X_mock = rng.normal(size=(60, 1764)).astype(np.float32)
y_mock = np.array([0] * 30 + [1] * 30)

clf_bin, scaler_bin = train_svm(X_mock, y_mock, C=1.0, kernel="rbf")
eval_res = evaluate_svm(clf_bin, scaler_bin, X_mock, y_mock)
print(f"Binary SVM Training Accuracy: {eval_res['accuracy']:.4f}")
print(f"Binary SVM Training Macro-F1: {eval_res['f1_macro']:.4f}")

## 3. Phân Tích Ma Trận Nhầm Lẫn & Các Cặp Lớp Khó (Error Analysis)

In [ ]:
diag = analyze_confusion_and_per_class_f1(y_mock, eval_res["y_pred"], class_names=["Background", "Traffic_Sign"])
print(f"Tổng số mẫu đánh giá: {diag['total_samples']}")
print(f"Macro-F1 đạt được: {diag['macro_f1']:.4f}")

## 4. Huấn Luyện & Lưu Mô Hình Trên Ảnh Mẫu (Export Models)

In [ ]:
# Trích xuất đặc trưng từ ảnh mẫu trong data/raw/images/
config, project_root, _ = load_config()
images_dir = project_root / config["paths"]["data_raw"] / "images"
labels_dir = project_root / config["paths"]["data_raw"] / "labels"

sign_feats, sign_labels = [], []
for p in images_dir.glob("*.jpg"):
    img = load_image(p)
    lbl = labels_dir / f"{p.stem}.txt"
    if not lbl.is_file():
        continue
    for b in read_label_boxes(lbl, img.shape):
        crop = img[max(0, b['y1']):min(img.shape[0], b['y2']), max(0, b['x1']):min(img.shape[1], b['x2'])]
        if crop.size > 0:
            feat, _ = extract_hog_features(crop)
            sign_feats.append(feat)
            sign_labels.append(b['class'])

if sign_feats:
    # Tạo mẫu nền ngẫu nhiên
    bg_feats = []
    sample_img = load_image(list(images_dir.glob('*.jpg'))[0])
    H, W = sample_img.shape[:2]
    for _ in range(len(sign_feats)):
        rx, ry = rng.integers(0, max(1, W - 64)), rng.integers(0, max(1, H - 64))
        bg_crop = sample_img[ry:ry+64, rx:rx+64]
        feat, _ = extract_hog_features(bg_crop)
        bg_feats.append(feat)

    X_b = np.vstack([sign_feats, bg_feats])
    y_b = np.array([1] * len(sign_feats) + [0] * len(bg_feats))
    clf_b, sc_b = train_svm(X_b, y_b, C=1.0)
    save_model(clf_b, sc_b, project_root / "outputs/models/svm_binary.joblib")

    X_m = np.array(sign_feats)
    y_m = np.array(sign_labels)
    clf_m, sc_m = train_svm(X_m, y_m, C=2.0)
    save_model(clf_m, sc_m, project_root / "outputs/models/svm_multiclass.joblib")
    print("Đã lưu thành công mô hình vào outputs/models/")